# Diffusion-TS data inspection

Walk through the datasets in `./Data/datasets/` the way the pipeline sees them:
raw CSV → `MinMaxScaler` → windowed samples → (for test) mask.

Datasets:
- `ETTh.csv` — 7 features, hourly; config drops the date column (`real_datasets.py:134-135`)
- `energy_data.csv` — multivariate appliance energy
- `stock_data.csv` — 6 features
- `fMRI/sim4.mat` — different loader (`fMRIDataset.read_data`)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import io
from sklearn.preprocessing import MinMaxScaler

# Notebook lives at repo root; make sure relative paths resolve there.
REPO_ROOT = os.path.abspath(os.path.dirname("__file__") or ".")
if os.path.basename(os.getcwd()) != "Diffusion-TS":
    os.chdir(REPO_ROOT)
DATA_DIR = "./Data/datasets"
print("cwd:", os.getcwd())
print("files:", sorted(os.listdir(DATA_DIR)))

cwd: /home/kexin/Repos/data-gen/Diffusion-TS
files: ['ETTh.csv', 'energy_data.csv', 'fMRI', 'stock_data.csv']


# ETTh

In [2]:
# --- ETTh: raw look ---
etth_path = f"{DATA_DIR}/ETTh.csv"
etth_raw = pd.read_csv(etth_path, header=0)

print("shape:", etth_raw.shape)
print("\ndtypes:\n", etth_raw.dtypes)
print("\nmissing per column:\n", etth_raw.isna().sum())
etth_raw.head()

shape: (17420, 8)

dtypes:
 date     object
HUFL    float64
HULL    float64
MUFL    float64
MULL    float64
LUFL    float64
LULL    float64
OT      float64
dtype: object

missing per column:
 date    0
HUFL    0
HULL    0
MUFL    0
MULL    0
LUFL    0
LULL    0
OT      0
dtype: int64


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
3,2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
4,2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000


# Stock

In [3]:
# --- Stock: raw look ---
stock_data = f"{DATA_DIR}/stock_data.csv"
stock_raw = pd.read_csv(stock_data, header=0)

print("shape:", stock_raw.shape)
print("\ndtypes:\n", stock_raw.dtypes)
print("\nmissing per column:\n", stock_raw.isna().sum())
stock_raw.head()

shape: (3685, 6)

dtypes:
 Open         float64
High         float64
Low          float64
Close        float64
Adj_Close    float64
Volume         int64
dtype: object

missing per column:
 Open         0
High         0
Low          0
Close        0
Adj_Close    0
Volume       0
dtype: int64


,Open,High,Low,Close,Adj_Close,Volume
0,49.676899,51.693783,47.669952,49.845802,49.845802,44994500
1,50.178635,54.187561,49.925285,53.805050,53.805050,23005800
2,55.017166,56.373344,54.172661,54.346527,54.346527,18393200
3,55.260582,55.439419,51.450363,52.096165,52.096165,15361800
4,52.140873,53.651051,51.604362,52.657513,52.657513,9257400


# Energy

In [4]:
# --- energy_data: raw look ---
energy_data = f"{DATA_DIR}/energy_data.csv"
energy_raw = pd.read_csv(energy_data, header=0)

print("shape:", energy_raw.shape)
print("\ndtypes:\n", energy_raw.dtypes)
print("\nmissing per column:\n", energy_raw.isna().sum())
energy_raw.head()

shape: (19735, 28)

dtypes:
 Appliances       int64
lights           int64
T1             float64
RH_1           float64
T2             float64
RH_2           float64
T3             float64
RH_3           float64
T4             float64
RH_4           float64
T5             float64
RH_5           float64
T6             float64
RH_6           float64
T7             float64
RH_7           float64
T8             float64
RH_8           float64
T9             float64
RH_9           float64
T_out          float64
Press_mm_hg    float64
RH_out         float64
Windspeed      float64
Visibility     float64
Tdewpoint      float64
rv1            float64
rv2            float64
dtype: object

missing per column:
 Appliances     0
lights         0
T1             0
RH_1           0
T2             0
RH_2           0
T3             0
RH_3           0
T4             0
RH_4           0
T5             0
RH_5           0
T6             0
RH_6           0
T7             0
RH_7           0
T8             0
RH

,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,45.566667,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,45.992500,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,45.890000,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,45.723333,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,45.530000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


# fMRI

In [5]:
# --- fMRI: load the .mat file ---
# Mirrors fMRIDataset.read_data (Utils/Data_utils/real_datasets.py:181).
# Only 'ts' is used for training; other keys are metadata about the simulation.
fmri_mat = io.loadmat(f"{DATA_DIR}/fMRI/sim4.mat")

print("user keys:", [k for k in fmri_mat if not k.startswith("__")])
for k, v in fmri_mat.items():
    if k.startswith("__"):
        continue
    print(f"  {k:12s} dtype={v.dtype}  shape={v.shape}")

fmri_ts = fmri_mat["ts"]  # (10000, 50) — 50 ROIs, 10000 timepoints
fmri_net = fmri_mat["net"]  # (50, 50, 50) — ground-truth connectivity per subject
print("\nts[:3, :5]:\n", fmri_ts[:3, :5])
print("\nNnodes/Nsubjects/Ntimepoints:",
      int(fmri_mat["Nnodes"]), int(fmri_mat["Nsubjects"]), int(fmri_mat["Ntimepoints"]))

user keys: ['ts', 'net', 'Nnodes', 'Nsubjects', 'Ntimepoints']
  ts           dtype=float64  shape=(10000, 50)
  net          dtype=float64  shape=(50, 50, 50)
  Nnodes       dtype=uint8  shape=(1, 1)
  Nsubjects    dtype=uint8  shape=(1, 1)
  Ntimepoints  dtype=uint8  shape=(1, 1)

ts[:3, :5]:
 [[-1.9256301  -2.04509448 -1.44860076 -2.81116244 -0.48910805]
 [-0.65153503 -2.84865324 -1.95699317 -1.19830735 -0.40468459]
 [ 1.40210147 -0.74846544 -0.62276712 -0.47352736  1.26533856]]

Nnodes/Nsubjects/Ntimepoints: 50 50 200


# Sine

In [6]:
# --- Sine: generate and inspect ---
# SineDataset synthesizes data on the fly — no file to load.
# Each feature is sin(freq*t + phase) with freq, phase ~ U(0, 0.1).
# Config default: num=10000, dim=5, window=24. Using num=200 here just to see the shape.
import sys
if "." not in sys.path:
    sys.path.insert(0, ".")
from Utils.Data_utils.sine_dataset import SineDataset

sine_ds = SineDataset(num=200, dim=5, window=24, save2npy=False, seed=123)
print("rawdata:", sine_ds.rawdata.shape, sine_ds.rawdata.dtype, "range:", sine_ds.rawdata.min(), sine_ds.rawdata.max())
print("samples:", sine_ds.samples.shape, "range:", sine_ds.samples.min(), sine_ds.samples.max())
print("one sample [0, :, 0]:", sine_ds.samples[0, :, 0])

/home/kexin/Repos/data-gen/Diffusion-TS/.venv/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Sampling sine-dataset: 100%|██████████| 200/200 [00:00<00:00, 8833.65it/s]

rawdata: (200, 24, 5) float64 range: 0.5000060050744486 0.999999996760204
samples: (200, 24, 5) range: 1.2010148897179107e-05 0.999999993520408
one sample [0, :, 0]: [0.02861003 0.09810281 0.16711991 0.23532669 0.30239244 0.36799197
 0.4318072  0.49352872 0.55285724 0.60950511 0.66319765 0.71367453
 0.76069098 0.80401905 0.84344865 0.87878859 0.90986751 0.93653474
 0.95866095 0.97613888 0.98888377 0.99683382 0.9999505  0.99821868]


# Mujoco

In [7]:
# --- MuJoCo: run the simulator and inspect ---
# MuJoCoDataset drives dm_control's hopper-stand env. Each sample = random init + `window` physics steps,
# recording qpos in the first half of features and qvel in the second half.
# Config default: num=10000, dim=14, window=24. Using num=200 here to keep it fast.
from Utils.Data_utils.mujoco_dataset import MuJoCoDataset

mj_ds = MuJoCoDataset(num=200, dim=14, window=24, save2npy=False, seed=123)
print("rawdata:", mj_ds.rawdata.shape, mj_ds.rawdata.dtype, "range:", mj_ds.rawdata.min(), mj_ds.rawdata.max())
print("samples:", mj_ds.samples.shape, "range:", mj_ds.samples.min(), mj_ds.samples.max())
print("qpos feature 0 over time, sample 0:", mj_ds.samples[0, :, 0])
print("qvel feature 7 over time, sample 0:", mj_ds.samples[0, :, 7])

/home/kexin/Repos/data-gen/Diffusion-TS/.venv/lib/python3.8/site-packages/glfw/__init__.py:917: GLFWError: (65550) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)


rawdata: (200, 24, 14) float64 range: -40.091314673736235 52.495027262174844
samples: (200, 24, 14) range: -1.0 1.0
qpos feature 0 over time, sample 0: [0.09933148 0.10997902 0.12042188 0.13089051 0.14148218 0.15220068
 0.1630494  0.17403121 0.18514847 0.196403   0.20779597 0.21932796
 0.23099889 0.24280803 0.25475401 0.26683486 0.27904798 0.29139029
 0.30385819 0.3164477  0.3291545  0.34197403 0.3549016  0.36793245]
qvel feature 7 over time, sample 0: [0.28525323 0.2717773  0.2669363  0.26754597 0.27045583 0.2734557
 0.2765353  0.27968322 0.28288702 0.28613329 0.2894078  0.29269572
 0.29598181 0.29925072 0.30248729 0.30567686 0.3088056  0.31186084
 0.31483131 0.31770743 0.32048153 0.32314794 0.32570314 0.32814577]


In [ ]:
# --- Parse configs: load all YAMLs and compare key fields ---
# Diffusion-TS uses two helpers in Utils/io_utils.py:
#   load_yaml_config(path)       -> nested dict (yaml.full_load)
#   instantiate_from_config(cfg) -> imports cfg['target'] and calls it with **cfg['params']
from Utils.io_utils import load_yaml_config, instantiate_from_config

CONFIG_DIR = "./Config"
names = ["etth", "energy", "stocks", "fmri", "sines", "mujoco"]
configs = {n: load_yaml_config(f"{CONFIG_DIR}/{n}.yaml") for n in names}

rows = []
for n, cfg in configs.items():
    mp = cfg["model"]["params"]
    train = cfg["dataloader"]["train_dataset"]["params"]
    rows.append({
        "config": n,
        "dataset_class": cfg["dataloader"]["train_dataset"]["target"].rsplit(".", 1)[-1],
        "seq_length": mp["seq_length"],
        "feature_size": mp["feature_size"],
        "timesteps": mp["timesteps"],
        "max_epochs": cfg["solver"]["max_epochs"],
        "batch_size": cfg["dataloader"]["batch_size"],
        "data_root": train.get("data_root"),  # file-based datasets only
        "num": train.get("num"),              # generator-based only
        "dim": train.get("dim"),              # generator-based only
    })
summary = pd.DataFrame(rows).set_index("config")
summary

In [ ]:
# --- Instantiate ETTh straight from its config (no hardcoded params) ---
# This is what Data/build_dataloader.py does internally.
# We override save2npy=False so inspection doesn't write to OUTPUT/.
# In the real pipeline, build_dataloader() also injects args.save_dir as output_dir.
etth_cfg = configs["etth"]["dataloader"]["train_dataset"]
etth_cfg_override = {
    **etth_cfg,
    "params": {**etth_cfg["params"], "save2npy": False},
}
etth_ds = instantiate_from_config(etth_cfg_override)

print("target:        ", etth_cfg["target"])
print("rawdata shape: ", etth_ds.rawdata.shape, " (continuous series, 2D)")
print("samples shape: ", etth_ds.samples.shape, " (sliding windows, 3D)")
print("window:        ", etth_ds.window)
print("var_num:       ", etth_ds.var_num)
print("scaler min/max:\n ", etth_ds.scaler.data_min_, "\n ", etth_ds.scaler.data_max_)